In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
from collections import Counter
import torch.optim as optim

nltk.download('punkt')

# =========================
# 1. LOAD DATA
# =========================

dataset = load_dataset("squad")
data = dataset["train"]

pairs = [(item["question"], item["answers"]["text"][0]) for item in data]
pairs = pairs[:5000]

print("Total pairs:", len(pairs))

# =========================
# 2. TOKENIZER
# =========================

def tokenize(text):
    return nltk.word_tokenize(text.lower())

# =========================
# 3. VOCAB
# =========================

counter = Counter()
for q, a in pairs:
    counter.update(tokenize(q))
    counter.update(tokenize(a))

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

vocab_size = 15000
most_common = counter.most_common(vocab_size - 4)

idx2word = [PAD, UNK, SOS, EOS] + [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(idx2word)}

print("Vocab:", len(word2idx))

# =========================
# 4. ENCODE
# =========================

MAX_LEN = 30

def encode(text):
    tokens = [SOS] + tokenize(text) + [EOS]
    ids = [word2idx.get(t, word2idx[UNK]) for t in tokens]

    if len(ids) < MAX_LEN:
        ids += [word2idx[PAD]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

# =========================
# 5. DATASET
# =========================

class QADataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, a = self.pairs[idx]
        return torch.tensor(encode(q)), torch.tensor(encode(a))

dataset = QADataset(pairs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# =========================
# 6. MODEL (ATTENTION)
# =========================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        outputs, hidden = self.rnn(x)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, hidden, encoder_outputs):
        x = self.embedding(x)

        hidden_repeat = hidden.permute(1, 0, 2).repeat(1, encoder_outputs.size(1), 1)
        energy = torch.tanh(self.attn(torch.cat((hidden_repeat, encoder_outputs), dim=2)))

        attention = self.softmax(energy.sum(dim=2))

        context = torch.bmm(attention.unsqueeze(1), encoder_outputs)
        context = context.repeat(1, x.size(1), 1)

        x = torch.cat((x, context), dim=2)

        output, hidden = self.rnn(x, hidden)
        output = self.fc(output)

        return output, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        encoder_outputs, hidden = self.encoder(src)
        output, _ = self.decoder(trg, hidden, encoder_outputs)
        return output


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        encoder_outputs, hidden = self.encoder(src)
        output, _ = self.decoder(trg, hidden, encoder_outputs)
        return output

encoder = Encoder(len(word2idx), 64, 128)
decoder = Decoder(len(word2idx), 64, 128)

model = Seq2Seq(encoder, decoder)

# =========================
# 7. TRAINING
# =========================

criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD])
optimizer = optim.Adam(model.parameters(), lr=0.0001)

epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for questions, answers in loader:

        optimizer.zero_grad()

        outputs = model(questions, answers)
        outputs = outputs[:, :-1, :]
        target = answers[:, 1:]

        outputs = outputs.reshape(-1, outputs.shape[-1])
        target = target.reshape(-1)

        loss = criterion(outputs, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# =========================
# 8. PREDICTION
# =========================

def predict_sentence(text, max_len=10):
    model.eval()

    
    encoded = encode(text)
    input_tensor = torch.tensor([encoded])

    
    with torch.no_grad():
         encoder_outputs, hidden = model.encoder(input_tensor)

    
    current_token = torch.tensor([[word2idx["<SOS>"]]])
    result = []

    
    for _ in range(max_len):
        with torch.no_grad():
            output, hidden = model.decoder(current_token, hidden, encoder_outputs)

            
            temperature = 0.6
            probs = torch.softmax(output[:, -1, :] / temperature, dim=1)

            top_k = 5
            values, indices = torch.topk(probs, top_k)

            values = values.squeeze()
            indices = indices.squeeze()

            values = values / torch.sum(values)

            predicted_index = indices[torch.multinomial(values, 1)].item()

        word = idx2word[predicted_index]

        bad_starts = ["and"]

        if len(result) == 0 and word in bad_starts:
            continue


        if len(result) > 0 and word == result[-1]:
            continue

        
        if word == "<EOS>":
            break

        if word not in ["<PAD>", "<SOS>"]:
            result.append(word)

        
        current_token = torch.tensor([[predicted_index]])

    if len(result) == 0:
        return "I don't know yet"

    return " ".join(result)

# =========================
# 9. TEST
# =========================

print("Example Predictions:\n")

print(predict_sentence("Who is he?"))
print(predict_sentence("What is Python?"))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total pairs: 5000
Vocab: 8198
Epoch 1, Loss: 1242.5149
Epoch 2, Loss: 968.4535
Epoch 3, Loss: 926.7435
Epoch 4, Loss: 914.6408
Epoch 5, Loss: 905.6150
Epoch 6, Loss: 899.6101
Epoch 7, Loss: 891.2309
Epoch 8, Loss: 885.8352
Epoch 9, Loss: 878.7217
Epoch 10, Loss: 873.4189
Example Predictions:

a
the
